# Informe de Pruebas de Hipótesis Pareadas

Este documento presenta una comparación estadística entre tres funciones de pérdida del pipeline de regresión simbólica: `mse`, `sigmoid` y `match_count`.

**Objetivo:** evaluar si el método sigmoidal presenta mejor desempeño, en términos de éxito del pipeline y tiempo de ejecución.

## Diseño Experimental y Metodología

Este análisis compara tres funciones de pérdida del pipeline de regresión simbólica:
- `mse`
- `sigmoid`
- `match_count`

Cada caso de benchmark aparece evaluado en los tres métodos. 

### Pares comparados
- sigmoid vs mse
- sigmoid vs match_count
- mse vs match_count

### Métricas evaluadas por par
- `success` (éxito del pipeline): variable binaria (1 éxito, 0 no éxito).
- `time` (tiempo de ejecución en segundos): variable continua.

### Hipótesis por métrica (prueba t)
- Para `success`: H1 = método A > método B.
- Para `time`: H1 = método A < método B (A es más rápido).

Nivel de significancia: `alpha = 0.05`.

In [22]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats

ALPHA = 0.05
METHODS = ["mse", "sigmoid", "match_count"]
PAIRS = [("sigmoid", "mse"), ("sigmoid", "match_count"), ("mse", "match_count")]
METRICS = ["success", "time"]

## 1) Configuración del análisis

En esta sección se importan librerías y se definen constantes del experimento (`alpha`, métodos, pares y métricas).

In [23]:
def detect_tests_base() -> Path:
    candidates = [
        Path('.'),
        Path('src/tests'),
        Path('../tests'),
        Path('../src/tests'),
    ]
    for cand in candidates:
        if (
            (cand / 'test_mse' / 'evaluations.json').exists()
            and (cand / 'test_sigmoid' / 'evaluations.json').exists()
            and (cand / 'test_match_count' / 'evaluations.json').exists()
        ):
            return cand
    raise FileNotFoundError('No se encontró la carpeta de tests con evaluations.json.')

base = detect_tests_base()
print(f'Base de tests detectada: {base.resolve()}')

Base de tests detectada: /home/noel/Disco D/4to_Anno/tesis/Tesis/src/tests


## 2) Detección de rutas y carga de resultados

Se detecta automáticamente la carpeta de tests y se verifica que existan los tres archivos `evaluations.json` necesarios para la comparación.

In [24]:
files = {
    'mse': base / 'test_mse' / 'evaluations.json',
    'sigmoid': base / 'test_sigmoid' / 'evaluations.json',
    'match_count': base / 'test_match_count' / 'evaluations.json',
}

data = {k: json.loads(v.read_text(encoding='utf-8')) for k, v in files.items()}
maps = {k: {row['name']: row for row in rows} for k, rows in data.items()}

case_names = sorted(maps['mse'].keys())
for method in ['sigmoid', 'match_count']:
    missing = sorted(set(case_names) - set(maps[method].keys()))
    extra = sorted(set(maps[method].keys()) - set(case_names))
    if missing or extra:
        raise ValueError(
            f"Los casos de {method} no coinciden con mse. Faltan: {missing[:5]}, Extras: {extra[:5]}"
        )

common_cases = case_names
print(f'Casos usados en el análisis (sin filtrar): {len(common_cases)}')
print('Primeros 5 casos:', common_cases[:5])

Casos usados en el análisis (sin filtrar): 43
Primeros 5 casos: ['cubic_25_one_root', 'cubic_26_factored', 'cubic_27_quadratic_factor', 'cubic_28_depressed', 'cubic_29_two_params']


## 3) Preparación del dataset 

Se usan todos los casos del benchmark y se verifica consistencia entre métodos.


In [25]:
rows = []
for case in common_cases:
    row = {'name': case}
    for method in METHODS:
        d = maps[method][case]
        row[f'{method}_success'] = 1.0 if d.get('status') == 'success' else 0.0
        row[f'{method}_time'] = float(d.get('time_seconds', np.nan))
    rows.append(row)

df = pd.DataFrame(rows)
print(df)

                                name  mse_success    mse_time  \
0                  cubic_25_one_root          1.0   79.017790   
1                  cubic_26_factored          1.0  287.808061   
2          cubic_27_quadratic_factor          1.0   71.202750   
3                 cubic_28_depressed          0.0  219.127649   
4                cubic_29_two_params          1.0  337.823126   
5                    cubic_30_triple          1.0   74.109887   
6                    linear_01_basic          1.0  153.487449   
7                   linear_02_scaled          1.0   80.090852   
8                   linear_03_offset          1.0   91.528395   
9                 linear_04_fraction          1.0   74.545965   
10                linear_05_negative          1.0   66.460936   
11              linear_06_two_params          1.0   85.966053   
12           linear_07_scaled_params          1.0   86.715830   
13           linear_08_product_param          1.0   88.078700   
14                linear_

## 4) Pruebas de hipótesis individuales

En esta sección se realiza **cada prueba de hipótesis en un bloque de código independiente**.



In [26]:
def run_single_hypothesis_test(df: pd.DataFrame, method_a: str, method_b: str, metric: str, alpha: float = 0.05):
    col_a = f"{method_a}_{metric}"
    col_b = f"{method_b}_{metric}"

    a = df[col_a].to_numpy(dtype=float)
    b = df[col_b].to_numpy(dtype=float)
    diff = a - b

    # Se incluyen todos los pares; solo se excluyen NaN si existieran.
    valid = ~np.isnan(diff)
    a = a[valid]
    b = b[valid]
    diff = diff[valid]

    if metric == "success":
        alternative = "greater"
        h0 = f"mu({method_a}-{method_b}) <= 0"
        h1 = f"mu({method_a}-{method_b}) > 0"
    elif metric == "time":
        alternative = "less"
        h0 = f"mu({method_a}-{method_b}) >= 0"
        h1 = f"mu({method_a}-{method_b}) < 0"
    else:
        raise ValueError(f"Métrica no soportada: {metric}")

    if len(diff) < 2:
        print("No hay suficientes datos para ejecutar la prueba.")
        return {
            "pair": f"{method_a} vs {method_b}",
            "metric": metric,
            "n": len(diff),
            "t_stat": np.nan,
            "p_value": np.nan,
            "reject_h0": False,
            "mean_a": float(np.nanmean(a)) if len(a) else np.nan,
            "mean_b": float(np.nanmean(b)) if len(b) else np.nan,
            "mean_diff": float(np.nanmean(diff)) if len(diff) else np.nan,
            "h0": h0,
            "h1": h1,
        }

    # Caso especial: todas las diferencias iguales.
    if np.isclose(np.std(diff, ddof=1), 0.0):
        mean_diff = float(np.mean(diff))
        if alternative == "greater":
            p_value = 0.0 if mean_diff > 0 else 1.0
        else:
            p_value = 0.0 if mean_diff < 0 else 1.0
        t_stat = np.inf if p_value == 0.0 else 0.0
    else:
        res = stats.ttest_rel(a, b, alternative=alternative, nan_policy="omit")
        t_stat = float(res.statistic)
        p_value = float(res.pvalue)
        mean_diff = float(np.mean(diff))

    reject_h0 = p_value < alpha

    print(f"Comparación: {method_a} vs {method_b} | Métrica: {metric}")
    print(f"H0: {h0}")
    print(f"H1: {h1}")
    print(f"n pares: {len(diff)}")
    print(f"media({method_a}) = {np.mean(a):.6f}")
    print(f"media({method_b}) = {np.mean(b):.6f}")
    print(f"media({method_a} - {method_b}) = {mean_diff:.6f}")
    print(f"t = {t_stat:.6f}")
    print(f"p-value = {p_value:.6g}")
    print(f"Decisión (alpha={alpha}): {'RECHAZAR H0' if reject_h0 else 'NO rechazar H0'}")

    return {
        "pair": f"{method_a} vs {method_b}",
        "metric": metric,
        "n": len(diff),
        "t_stat": t_stat,
        "p_value": p_value,
        "reject_h0": reject_h0,
        "mean_a": float(np.mean(a)),
        "mean_b": float(np.mean(b)),
        "mean_diff": mean_diff,
        "h0": h0,
        "h1": h1,
    }

## 5) Pruebas de hipótesis 

A continuación se ejecutan 6 pruebas t pareadas , una por cada combinación de par y métrica.

En cada bloque se reporta:
- hipótesis nula (H0)
- hipótesis alternativa (H1)
- estadísticos de prueba
- decisión final (rechazar o no rechazar H0).

### Test 1: Sigmoid vs MSE en éxito del pipeline

**Objetivo:** evaluar si `sigmoid` tiene mayor tasa de éxito que `mse`.

- H0: $\mu(\text{sigmoid} - \text{mse}) \le 0$
- H1: $\mu(\text{sigmoid} - \text{mse}) > 0$

In [33]:
test_1 = run_single_hypothesis_test(
    df=df,
    method_a='sigmoid',
    method_b='mse',
    metric='success',
    alpha=ALPHA,
)

Comparación: sigmoid vs mse | Métrica: success
H0: mu(sigmoid-mse) <= 0
H1: mu(sigmoid-mse) > 0
n pares: 43
media(sigmoid) = 1.000000
media(mse) = 0.581395
media(sigmoid - mse) = 0.418605
t = 5.499091
p-value = 1.03967e-06
Decisión (alpha=0.05): RECHAZAR H0


### Test 2: Sigmoid vs MSE en tiempo de ejecución

**Objetivo:** evaluar si `sigmoid` es más rápido que `mse` (menor tiempo promedio).

- H0: $\mu(\text{sigmoid} - \text{mse}) \ge 0$
- H1: $\mu(\text{sigmoid} - \text{mse}) < 0$

In [28]:
test_2 = run_single_hypothesis_test(
    df=df,
    method_a='sigmoid',
    method_b='mse',
    metric='time',
    alpha=ALPHA,
)

Comparación: sigmoid vs mse | Métrica: time
H0: mu(sigmoid-mse) >= 0
H1: mu(sigmoid-mse) < 0
n pares: 43
media(sigmoid) = 184.163437
media(mse) = 209.651931
media(sigmoid - mse) = -25.488493
t = -1.112133
p-value = 0.136203
Decisión (alpha=0.05): NO rechazar H0


### Test 3: Sigmoid vs Match Count en éxito del pipeline

**Objetivo:** evaluar si `sigmoid` tiene mayor tasa de éxito que `match_count`.

- H0: $\mu(\text{sigmoid} - \text{match\_count}) \le 0$
- H1: $\mu(\text{sigmoid} - \text{match\_count}) > 0$

In [34]:
test_3 = run_single_hypothesis_test(
    df=df,
    method_a='sigmoid',
    method_b='match_count',
    metric='success',
    alpha=ALPHA,
)

Comparación: sigmoid vs match_count | Métrica: success
H0: mu(sigmoid-match_count) <= 0
H1: mu(sigmoid-match_count) > 0
n pares: 43
media(sigmoid) = 1.000000
media(match_count) = 0.953488
media(sigmoid - match_count) = 0.046512
t = 1.431356
p-value = 0.0798639
Decisión (alpha=0.05): NO rechazar H0


### Test 4: Sigmoid vs Match Count en tiempo de ejecución

**Objetivo:** evaluar si `sigmoid` es más rápido que `match_count`.

- H0: $\mu(\text{sigmoid} - \text{match\_count}) \ge 0$
- H1: $\mu(\text{sigmoid} - \text{match\_count}) < 0$

In [30]:
test_4 = run_single_hypothesis_test(
    df=df,
    method_a='sigmoid',
    method_b='match_count',
    metric='time',
    alpha=ALPHA,
)

Comparación: sigmoid vs match_count | Métrica: time
H0: mu(sigmoid-match_count) >= 0
H1: mu(sigmoid-match_count) < 0
n pares: 43
media(sigmoid) = 184.163437
media(match_count) = 432.485792
media(sigmoid - match_count) = -248.322355
t = -4.943664
p-value = 6.38983e-06
Decisión (alpha=0.05): RECHAZAR H0


### Test 5: MSE vs Match Count en éxito del pipeline

**Objetivo:** evaluar si `mse` tiene mayor tasa de éxito que `match_count`.

- H0: $\mu(\text{mse} - \text{match\_count}) \le 0$
- H1: $\mu(\text{mse} - \text{match\_count}) > 0$

In [31]:
test_5 = run_single_hypothesis_test(
    df=df,
    method_a='mse',
    method_b='match_count',
    metric='success',
    alpha=ALPHA,
)

Comparación: mse vs match_count | Métrica: success
H0: mu(mse-match_count) <= 0
H1: mu(mse-match_count) > 0
n pares: 43
media(mse) = 0.581395
media(match_count) = 0.953488
media(mse - match_count) = -0.372093
t = -4.219162
p-value = 0.999936
Decisión (alpha=0.05): NO rechazar H0


### Test 6: MSE vs Match Count en tiempo de ejecución

**Objetivo:** evaluar si `mse` es más rápido que `match_count`.

- H0: $\mu(\text{mse} - \text{match\_count}) \ge 0$
- H1: $\mu(\text{mse} - \text{match\_count}) < 0$

In [35]:
test_6 = run_single_hypothesis_test(
    df=df,
    method_a='mse',
    method_b='match_count',
    metric='time',
    alpha=ALPHA,
)

Comparación: mse vs match_count | Métrica: time
H0: mu(mse-match_count) >= 0
H1: mu(mse-match_count) < 0
n pares: 43
media(mse) = 209.651931
media(match_count) = 432.485792
media(mse - match_count) = -222.833861
t = -4.504836
p-value = 2.61004e-05
Decisión (alpha=0.05): RECHAZAR H0


## Resumen final 
El uso de la funcion sigmoidal mejora notablemente el éxito del pipeline, aunque no es más rápida que match_count. Match_count no muestra mejoras significativas frente a MSE.
